# Step 1 — Cleaning

Turning the raw *Online Retail II* file into a clean sales table. Every cleaning rule exists because of something found in the data; the full reasoning, with numbers, is in [NOTES.md](../NOTES.md). This notebook runs the cleaner and shows what each rule removed.

The cleaner produces two views: `sales` (every valid sale, anonymous rows included) for basket analysis, and `customer_sales` (the subset with a Customer ID) for the customer work.

In [1]:
# Setup: run from the project root so the data directory and results/
# resolve exactly as they do in main.py, and the step modules import.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import pandas as pd
from IPython.display import Image, display

DATA_DIR = "kaggle_customer_intelligence"


In [2]:
from retail_cleaner import RetailCleaner

cleaner = RetailCleaner(f"{DATA_DIR}/online_retail_II.csv").run()
cleaner.save(DATA_DIR)

CLEANING LOG
Loaded 1,067,371 rows from kaggle_customer_intelligence/online_retail_II.csv

drop_duplicates                   1,067,371 ->  1,033,036  (removed 34,335)
remove_non_sales                  1,033,036 ->  1,007,912  (removed 25,124)

Dropping these non-product codes (top 20 by rows) — check they are all postage/fees/adjustments:
    POST              1,851
    DOT               1,415
    M                   851
    C2                  267
    ADJUST               36
    BANK CHARGES         34
    DCGS0058             30
    GIFT_0001_20         26
    GIFT_0001_30         24
    DCGSSGIRL            23
    DCGSSBOY             21
    PADS                 17
    DCGS0076             14
    GIFT_0001_10         14
    DCGS0003             13
    TEST001               9
    GIFT_0001_50          6
    GIFT_0001_40          5
    D                     5
    DCGS0069              5

remove_non_product_codes          1,007,912 ->  1,003,214  (removed 4,698)

SUMMARY OF CLEANED DAT

### What each step removed

The log above tracks the row count through every rule. Starting from **1,067,371** raw rows:

- **Duplicates:** 34,335 exact duplicate rows (3.22%). Quantity already records multiples, so identical rows are double entries.
- **Non-sales:** cancellations (invoices starting `C`), accounting adjustments (`A`), and any row with non-positive quantity or price.
- **Non-product codes:** postage, manual adjustments, fees, gift vouchers, and the like, which carry prices but aren't products.

Descriptions are then normalised to one canonical name per stock code, and a `Revenue = Quantity * Price` column is added.

In [3]:
cleaner.cleaning_log

,step,rows_before,rows_after
0,drop_duplicates,1067371,1033036
1,remove_non_sales,1033036,1007912
2,remove_non_product_codes,1007912,1003214


### The cleaned data

Roughly 87% of valid rows carry a Customer ID. The remaining anonymous rows are kept for basket analysis (where the invoice is the unit) but dropped from the customer work. The two views below feed the rest of the pipeline.

In [4]:
print('sales (all valid):     ', cleaner.sales.shape)
print('customer_sales (w/ ID): ', cleaner.customer_sales.shape)
cleaner.sales.head()

sales (all valid):      (1003214, 9)
customer_sales (w/ ID):  (776577, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


**Hands off to:** `cleaner.sales` for the EDA and basket steps, `cleaner.customer_sales` for RFM and segmentation.